In [ ]:
!pip install pandas openpyxl

In [ ]:
import pandas as pd

df = pd.read_excel('Х5.xlsx')

df.head()

In [ ]:
df_cut = df[['new_id', 'Трафик', 'Средний чек', 'Населенный пункт', 'Регион', 'Маркетплейсы, доставки, постаматы (100 м)', 'Медицинские уч. и аптеки (300 м)', 'Школы (300 м)', 'Остановки (300 м)', 'Продуктовые магазины (500 м)', 'Пятерочки (500 м)']].copy()

In [ ]:
import numpy as np
import pandas as pd
from category_encoders import TargetEncoder
from sklearn.model_selection import GroupKFold

In [ ]:
gkf = GroupKFold(n_splits=5)

In [ ]:
df_cut['Населенный пункт_fold_traffic'] = np.nan
df_cut['Регион_fold_traffic'] = np.nan
df_cut['Населенный пункт_fold_average_bill'] = np.nan
df_cut['Регион_fold_average_bill'] = np.nan

In [ ]:
for train_id, validation_id in gkf.split(df_cut, groups=df_cut['new_id']):
    
    X_train = df_cut.iloc[train_id][['Населенный пункт', 'Регион']]
    X_validation = df_cut.iloc[validation_id][['Населенный пункт', 'Регион']]
    y_train_traffic = df_cut.iloc[train_id]['Трафик']
    y_train_average_bill = df_cut.iloc[train_id]['Средний чек']
    
    TE_traffic = TargetEncoder(cols = ['Населенный пункт', 'Регион'], smoothing = 20)
    TE_traffic.fit(X_train, y_train_traffic)
    val_traffic = TE_traffic.transform(X_validation)
    
    df_cut.iloc[validation_id, df_cut.columns.get_loc('Населенный пункт_fold_traffic')] = val_traffic['Населенный пункт']
    df_cut.iloc[validation_id, df_cut.columns.get_loc('Регион_fold_traffic')] = val_traffic['Регион']
    
    TE_average_bill = TargetEncoder(cols = ['Населенный пункт', 'Регион'], smoothing = 20)
    TE_average_bill.fit(X_train, y_train_average_bill)
    val_average_bill = TE_average_bill.transform(X_validation)
    
    df_cut.iloc[validation_id, df_cut.columns.get_loc('Населенный пункт_fold_average_bill')] = val_average_bill['Населенный пункт']
    df_cut.iloc[validation_id, df_cut.columns.get_loc('Регион_fold_average_bill')] = val_average_bill['Регион']

In [ ]:
df_cut_traffic = df_cut[['new_id', 'Трафик', 'Средний чек', 'Населенный пункт_fold_traffic', 'Регион_fold_traffic', 'Маркетплейсы, доставки, постаматы (100 м)', 'Медицинские уч. и аптеки (300 м)', 'Школы (300 м)', 'Остановки (300 м)', 'Продуктовые магазины (500 м)', 'Пятерочки (500 м)']]

In [ ]:
df_cut_average_bill = df_cut[['new_id', 'Трафик', 'Средний чек', 'Населенный пункт_fold_average_bill', 'Регион_fold_average_bill', 'Маркетплейсы, доставки, постаматы (100 м)', 'Медицинские уч. и аптеки (300 м)', 'Школы (300 м)', 'Остановки (300 м)', 'Продуктовые магазины (500 м)', 'Пятерочки (500 м)']]

In [ ]:
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.pipeline import Pipeline
from sklearn.model_selection import train_test_split

In [ ]:
pca_param = df_cut_traffic[['Маркетплейсы, доставки, постаматы (100 м)', 'Медицинские уч. и аптеки (300 м)', 'Школы (300 м)', 'Остановки (300 м)', 'Продуктовые магазины (500 м)', 'Пятерочки (500 м)']]

In [ ]:
pca_param_train, pca_param_test = train_test_split(pca_param, test_size=0.3, random_state=598)

In [ ]:
pipeline = Pipeline([('scaler', StandardScaler()),('pca', PCA(n_components=4, random_state=598))])

In [ ]:
pipeline.fit(pca_param_train)

In [ ]:
counts = ['Маркетплейсы, доставки, постаматы (100 м)', 'Медицинские уч. и аптеки (300 м)', 'Школы (300 м)', 'Остановки (300 м)', 'Продуктовые магазины (500 м)', 'Пятерочки (500 м)']

In [ ]:
pca_average_bill_train = pipeline.transform(df_cut_average_bill.iloc[pca_param_train.index][counts])
pca_average_bill_test = pipeline.transform(df_cut_average_bill.iloc[pca_param_test.index][counts])
pca_traffic_train = pipeline.transform(df_cut_traffic.iloc[pca_param_train.index][counts])
pca_traffic_test = pipeline.transform(df_cut_traffic.iloc[pca_param_test.index][counts])

In [ ]:
sum(pipeline.named_steps['pca'].explained_variance_ratio_)

In [ ]:
pca_average_bill_res = pd.DataFrame(np.vstack([pca_average_bill_train, pca_average_bill_test]), columns=['pca_1', 'pca_2', 'pca_3', 'pca_4'], 
                            index=np.concatenate([pca_param_train.index, pca_param_test.index])).sort_index()

In [ ]:
pca_traffic_res = pd.DataFrame(np.vstack([pca_traffic_train, pca_traffic_test]), columns=['pca_1', 'pca_2', 'pca_3', 'pca_4'], 
                            index=np.concatenate([pca_param_train.index, pca_param_test.index])).sort_index()

In [ ]:
df_cut_average_bill = pd.concat([df_cut_average_bill, pca_average_bill_res], axis=1)
df_cut_traffic = pd.concat([df_cut_traffic, pca_traffic_res], axis=1)

In [ ]:
df_cut_average_bill = df_cut_average_bill.drop(counts, axis=1)
df_cut_traffic = df_cut_traffic.drop(counts, axis=1)